In [1]:
import numpy as np
import pandas as pd
import OpenEXR
import re
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageDraw

In [2]:
homePath = Path("/home/philipm/Desktop/corrected-datasets-folder/back_trees")

In [3]:
lightPath = "/media/philipm/FOUR/ThesisData/back_trees/"

In [4]:
trees = True

In [ ]:
for folder in homePath.glob("*"):
    for subfolder in folder.glob("*"):
        data = pd.DataFrame(columns=["grviIn", "meanRefR", "meanRefG", "illumR", "illumG", "blockIdx", "sunRoll", "sunPitch", "sunYaw", "camRoll", "camPitch", "camYaw"])

        test = Path(lightPath + folder.name + "/" + subfolder.name)
        
        with open(str(test) + "/EnvParams.txt", "r") as f:
            illuminanceFile = f.readlines()
        
        if trees:
            illumFiles = []
            
            with open(str(test) + r"/Light3.txt", "r") as f:
                lm3 = f.readlines()

            illumFiles = [lm3] * 5
            
            illuminances = []
            for i in range(0, 5):
                illuminancesSing = []
                for line in illumFiles[i]:
                    rg = []

                    raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', line)
                    
                    red = float(raw[len(raw)-3])
                    rg.append(red)
                    
                    green = float(raw[len(raw)-2])
                    rg.append(green)
                    
                    illuminancesSing.append(rg)
                illuminances.append(illuminancesSing)
        else:
            
            red_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[30])
            red = float(red_raw[0]) + float("0." + str(red_raw[1]))

            green_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[31])
            green = float(green_raw[0]) + float("0." + str(green_raw[1]))

            blue_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[32])
            blue = float(blue_raw[0]) + float("0." + str(blue_raw[1]))

            #debug
            print(red, green, blue)

        sunPos_raw = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', illuminanceFile[24])

        sunPos_roll = float(sunPos_raw[0])
        sunPos_pitch = float(str(sunPos_raw[1]) + "." + str(sunPos_raw[2]))
        sunPos_yaw = float(str(sunPos_raw[3]) + "." + str(sunPos_raw[4]))
        
        with open(str(test) + "/ObjectPoses.txt", "r") as f:
            camRotationFile = f.readlines()

        # this saves STRINGS
        rotations = []
        for i in range(0, len(camRotationFile)):
            step = re.findall(r'-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?', camRotationFile[i])
            step = step[len(step)-3 : len(step)]

            rotations.append(step)

        processedFolder = Path(str(subfolder) + "/Processed")

        files = sorted(
            processedFolder.glob("proc_*.npy"),
            key=lambda p: int(p.stem.split("_")[1])
        )
     
        for i in range(1, len(files)):
            blocks = np.load(str(files[i]))
            currRot = rotations[i]

            for j in range(0, len(blocks)):
                img = blocks[j]

                if (trees == False):
                    r_ref = (img[:, :, 0] * np.pi)/red 
                    g_ref = (img[:, :, 1] * np.pi)/green
    
                    grvi = (g_ref - r_ref)/(g_ref + r_ref + 0.00000001)
    
                    grvi_m = grvi.mean()
    
                    data.loc[len(data)] = [grvi_m, r_ref.mean(), g_ref.mean(), red, green, j, sunPos_roll, sunPos_pitch, sunPos_yaw, float(currRot[0]), float(currRot[1]), float(currRot[2])]

                else:
                    if (j>=0) and (j<4):
                        illumFile = illuminances[0]
                    elif (j>=4) and (j<8):
                        illumFile = illuminances[2]
                    elif (j>=8) and (j<12):
                        illumFile = illuminances[4]
                    elif (j>=12) and (j<16):
                        illumFile = illuminances[3]
                    else:
                        illumFile = illuminances[1]

                    rg_temp = illumFile[i-1]
                        
                    r_ref = (img[:, :, 0] * np.pi)/rg_temp[0]
                    g_ref = (img[:, :, 1] * np.pi)/rg_temp[1]
                    
                    grvi = (g_ref - r_ref)/(g_ref + r_ref + 0.00000001)
    
                    grvi_m = grvi.mean()
                    
                    data.loc[len(data)] = [grvi_m, r_ref.mean(), g_ref.mean(), rg_temp[0], rg_temp[1], j, sunPos_roll, sunPos_pitch, sunPos_yaw, float(currRot[0]), float(currRot[1]), float(currRot[2])]
                    
        data.to_csv(str(subfolder) + "/data-real.csv", index=False)

In [6]:
data

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw
0,0.311088,0.000171,0.000245,261.145,200.277,0.0,180.0,-19.798325,104.358398,-0.0,-20.058,-0.0
1,0.326886,0.000153,0.000235,261.145,200.277,1.0,180.0,-19.798325,104.358398,-0.0,-20.058,-0.0
2,0.306454,0.000181,0.000280,261.145,200.277,2.0,180.0,-19.798325,104.358398,-0.0,-20.058,-0.0
3,0.285081,0.000252,0.000376,261.145,200.277,3.0,180.0,-19.798325,104.358398,-0.0,-20.058,-0.0
4,0.292996,0.000251,0.000382,261.145,200.277,4.0,180.0,-19.798325,104.358398,-0.0,-20.058,-0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5935,-0.000211,0.371312,0.360556,0.854,1.033,15.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0
5936,-0.018895,0.375383,0.358165,0.854,1.033,16.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0
5937,0.008087,0.326700,0.316900,0.854,1.033,17.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0
5938,0.012527,0.273002,0.262333,0.854,1.033,18.0,180.0,-19.798325,104.358398,-0.0,-22.218,-180.0
